# Nhóm 3 Checkpoint Buổi 6 bản v2
Chạy trên Kaggle Python, CPU/None, Internet On; Add Input ealaxi/paysim1.
Chạy từ ô 01 đến ô 12, không bỏ qua bước lỗi. Mỗi lần chạy ô 01 tạo thư mục phiên mới.
Đã sửa đường dẫn Spark có dấu = và mã nhúng ở ô 05. Không cần thêm hotfix.
Các giá trị EXPECTED_SHA256/EXPECTED_BYTES lấy từ bộ kết quả của nhóm ngày 17/10/2026.
Nếu nguồn đổi snapshot, dừng để đối chiếu; không sửa raw hoặc lấy sample thay full.
Bộ này chưa có output chạy Kaggle được điền sẵn; dùng phiên chạy thật để lấy bằng chứng.


## Ô 01 khởi tạo thư mục và ghi nhận máy Kaggle


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, shutil, platform, csv, hashlib
from datetime import datetime, timezone
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
SOURCE_OVERRIDE = ''
EXPECTED_SHA256 = '16910f90577b0d981bf8ff289714510bb89bc71bff7d3f220f024e287e4eea6b'
EXPECTED_BYTES = 493534783
KAGGLE_DATASET_VERSION = None
PROJECT = Path('/kaggle/working/nhom3_buoi6') / ('session_' + RUN_ID)
for name in ['src','data/raw','data/sample','logs','metadata','docs','fixtures']:
    (PROJECT/name).mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT)
print('Working directory:', Path.cwd())
print('Python:', sys.version)
print('CPU:', os.cpu_count())
print('Free disk GB:', round(shutil.disk_usage(PROJECT).free/1e9, 2))
if Path('/proc/meminfo').exists():
    print(Path('/proc/meminfo').read_text().splitlines()[0])
print('Input mounts:', [p.name for p in Path('/kaggle/input').iterdir()])


## Ô 02 cài đúng PySpark cho notebook này


In [ ]:
from importlib.metadata import version, PackageNotFoundError
SPARK_VERSION = '4.1.1'
if sys.version_info < (3, 10):
    raise RuntimeError('Notebook nay yeu cau Python 3.10 tro len')
try:
    installed = version('pyspark')
except PackageNotFoundError:
    installed = None
if 'pyspark' in sys.modules and installed != SPARK_VERSION:
    raise RuntimeError('Restart session, khong import Spark truoc o 02')
if installed != SPARK_VERSION:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir',
                    'pyspark==' + SPARK_VERSION], check=True)
assert version('pyspark') == SPARK_VERSION
(PROJECT/'requirements.txt').write_text('pyspark=='+SPARK_VERSION+'\n', encoding='utf-8')
print('PYSPARK INSTALLED', version('pyspark'))


## Ô 03 kiểm Java và cấu hình Python cho worker

Dùng JDK 17 hoặc 21 đã có. Nếu chưa có, ô này thử cài OpenJDK 17 trong phiên Kaggle bằng apt-get, cần Internet. Không cài vào máy Windows của bạn.


In [ ]:
import re
def java_info(executable):
    try:
        proc = subprocess.run([str(executable), '-version'], capture_output=True, text=True)
        text = proc.stdout + proc.stderr
        match = re.search(r'version "(\d+)', text)
        if proc.returncode == 0 and match:
            return int(match.group(1)), text
    except OSError:
        pass
    return None, ''

candidates = []
if shutil.which('java'):
    candidates.append(Path(shutil.which('java')).resolve())
candidates += list(Path('/usr/lib/jvm').glob('*/bin/java'))
selected = next((p for p in candidates if java_info(p)[0] in (17,21)), None)
if selected is None:
    prefix = [] if os.geteuid() == 0 else ['sudo','-n']
    subprocess.run(prefix + ['apt-get','update'], check=True)
    subprocess.run(prefix + ['apt-get','install','-y','openjdk-17-jdk-headless'], check=True)
    selected = next((p for p in Path('/usr/lib/jvm').glob('*/bin/java')
                     if java_info(p)[0] == 17), None)
if selected is None:
    raise RuntimeError('Chua tim duoc JDK17/21. Xem traceback va Internet setting')
os.environ['JAVA_HOME'] = str(selected.resolve().parent.parent)
os.environ['PATH'] = str(selected.resolve().parent) + os.pathsep + os.environ['PATH']
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
os.environ['PYTHONIOENCODING'] = 'utf-8'
# This notebook uses one-host classic Spark, not Spark Connect.
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'
print('JAVA_HOME:', os.environ['JAVA_HOME'])
print(java_info(selected)[1])
print('Python worker:', os.environ['PYSPARK_PYTHON'])


## Ô 04 tìm CSV PaySim trong Input và ghi provenance

Nếu không tìm thấy, dùng Add Input gắn `ealaxi/paysim1` rồi chạy lại ô này. Không tự đổi sang Credit Card Fraud Detection vì schema khác. Khi biết version trong bảng Input, điền KAGGLE_DATASET_VERSION trước khi chạy.


In [ ]:
"""Find a specific PaySim snapshot, avoiding samples and duplicate-input ambiguity."""
import csv
import hashlib
from pathlib import Path

FIELDS = ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig',
          'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']


def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda: f.read(1024 * 1024), b''):
            h.update(b)
    return h.hexdigest()


def find_source(input_root, override, expected_hash, expected_bytes):
    if not expected_hash or len(expected_hash) != 64 or expected_bytes <= 0:
        raise ValueError('Set the expected SHA-256 and bytes from checkpoint source metadata')
    paths = [Path(override)] if override else sorted(Path(input_root).rglob('*.csv'))
    candidates = []
    for p in paths:
        if not p.is_file() or p.stat().st_size != expected_bytes:
            continue
        try:
            with p.open(encoding='utf-8-sig', newline='') as f:
                if next(csv.reader(f), None) == FIELDS:
                    candidates.append(p)
        except (OSError, UnicodeError, csv.Error):
            continue
    for p in candidates:
        actual = sha256_file(p)
        if actual == expected_hash:
            return p, actual
    raise ValueError('No matching FULL PaySim snapshot. Add Input ealaxi/paysim1; compare source metadata. Do not select the 1000-row sample or change expected hash without reviewing the source.')

SOURCE,SOURCE_HASH=find_source(Path('/kaggle/input'), SOURCE_OVERRIDE, EXPECTED_SHA256, EXPECTED_BYTES)
COLS=FIELDS
source_meta=dict(dataset='paysim',source_url='https://www.kaggle.com/datasets/ealaxi/paysim1',
 publisher='Edgar Lopez-Rojas (ealaxi)',source_path=str(SOURCE),
 accessed_at_utc=datetime.now(timezone.utc).isoformat(),kaggle_version=KAGGLE_DATASET_VERSION,
 bytes=SOURCE.stat().st_size,snapshot_sha256=SOURCE_HASH,license='CC BY-SA 4.0',
 note='Expected bytes/hash match the group snapshot; access time is not original download time.')
(PROJECT/'metadata/source.json').write_text(json.dumps(source_meta,indent=2),encoding='utf-8')
print(json.dumps(source_meta,indent=2))


## Ô 05 tạo script dự án và hàm chạy có log

Ô này xuất mã Python ra `src/` để nộp và đưa vào Git. Dương review ingestion, Dũng review verify/sample, My review env/Spark. Không phải cài lại bộ Windows cũ.


In [ ]:
SCRIPT_FILES = {'env_check.py': "import platform\nimport subprocess\nfrom pyspark.sql import SparkSession\n\nprint('Python:', platform.python_version(), flush=True)\nsubprocess.run(['java', '-version'], check=True)\nspark = SparkSession.builder.master('local[2]').appName('nhom3-env-check').getOrCreate()\ntry:\n    print('Spark:', spark.version)\n    actual = spark.range(1_000_000).count()\n    assert actual == 1_000_000, actual\n    print('Count:', actual)\n    print('ENV PASS')\nfinally:\n    spark.stop()\n", 'ingest_file.py': '"""Preserve source bytes; every accepted invocation creates a separate run."""\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport re\nimport shutil\nimport sys\nimport uuid\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nCOLUMNS = [\'step\', \'type\', \'amount\', \'nameOrig\', \'oldbalanceOrg\',\n           \'newbalanceOrig\', \'nameDest\', \'oldbalanceDest\',\n           \'newbalanceDest\', \'isFraud\', \'isFlaggedFraud\']\n\ndef digest(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\'--source\', required=True)\n    ap.add_argument(\'--dataset\', required=True)\n    ap.add_argument(\'--raw-root\', default=\'data/raw\')\n    ap.add_argument(\'--source-url\', default=\'https://www.kaggle.com/datasets/ealaxi/paysim1\')\n    ap.add_argument(\'--source-kind\', choices=[\'full_snapshot\', \'derived_sample\'], default=\'full_snapshot\')\n    a = ap.parse_args()\n    now = datetime.now(timezone.utc)\n    rid = now.strftime(\'%Y%m%dT%H%M%S%fZ\') + \'_\' + uuid.uuid4().hex[:8]\n    source = Path(a.source).resolve()\n    attempt = Path(\'metadata/attempts\') / (rid + \'.json\')\n    attempt.parent.mkdir(parents=True, exist_ok=True)\n    log = Path(\'logs\') / (\'ingest_\' + rid + \'.txt\')\n    log.parent.mkdir(parents=True, exist_ok=True)\n    record = dict(run_id=rid, dataset=a.dataset, source=str(source),\n                  source_url=a.source_url, source_kind=a.source_kind,\n                  ingested_at_utc=now.isoformat(), status=\'failed\')\n    stage = None\n    manifest = None\n    try:\n        if not re.fullmatch(r\'[a-z0-9_]+\', a.dataset):\n            raise ValueError(\'dataset: only a-z, 0-9, underscore\')\n        if not source.is_file() or source.stat().st_size == 0:\n            raise ValueError(\'source missing or empty\')\n        if source.suffix.lower() != \'.csv\':\n            raise ValueError(\'expected extracted .csv\')\n        before = source.stat()\n        with source.open(encoding=\'utf-8-sig\', newline=\'\') as f:\n            reader = csv.reader(f)\n            if next(reader, None) != COLUMNS:\n                raise ValueError(\'header must match 11 PaySim columns in order\')\n            checked = 0\n            for row in reader:\n                if len(row) != 11:\n                    raise ValueError(\'sample row must contain 11 fields\')\n                checked += 1\n                if checked == 20:\n                    break\n            if checked == 0:\n                raise ValueError(\'header-only CSV\')\n        folder = (Path(a.raw_root).resolve() / a.dataset /\n                  (\'ingest_date=\' + now.strftime(\'%Y-%m-%d\')) / (\'run_\' + rid))\n        folder.mkdir(parents=True, exist_ok=False)\n        target = folder / source.name\n        stage = folder / (source.name + \'.part\')\n        with source.open(\'rb\') as src, stage.open(\'xb\') as dst:\n            shutil.copyfileobj(src, dst, length=1024 * 1024)\n        fingerprint = digest(source)\n        after = source.stat()\n        if (after.st_size, after.st_mtime_ns) != (before.st_size, before.st_mtime_ns):\n            raise ValueError(\'source changed while ingesting\')\n        if stage.stat().st_size != before.st_size or digest(stage) != fingerprint:\n            raise ValueError(\'copy bytes/hash mismatch\')\n        stage.replace(target)\n        record.update(status=\'success\', bytes=target.stat().st_size,\n                      sha256=fingerprint, destination=str(target),\n                      sample_rows_checked=checked, data_rows=None,\n                      validation=\'header and first 20 rows; full Spark check separate\',\n                      rerun_policy=\'new run per invocation; same hash means duplicate snapshot\')\n        manifest = folder / \'manifest.json\'\n        # Publish acceptance marker only after trace files have been written.\n        payload = json.dumps(record, ensure_ascii=False, indent=2)\n        log.write_text(payload, encoding=\'utf-8\')\n        attempt.write_text(payload, encoding=\'utf-8\')\n        with manifest.open(\'x\', encoding=\'utf-8\') as f:\n            f.write(payload)\n        print(\'SUCCESS manifest=\' + str(manifest))\n        return 0\n    except Exception as e:\n        if stage is not None and stage.exists():\n            stage.unlink()\n        if manifest is not None and manifest.exists():\n            manifest.unlink()  # only this invocation\'s unfinished acceptance marker\n        record.update(status=\'failed\', error=type(e).__name__ + \': \' + str(e))\n        payload = json.dumps(record, ensure_ascii=False, indent=2)\n        log.write_text(payload, encoding=\'utf-8\')\n        attempt.write_text(payload, encoding=\'utf-8\')\n        print(\'FAILED \' + record[\'error\'], file=sys.stderr)\n        return 1\n\nif __name__ == \'__main__\':\n    raise SystemExit(main())\n', 'inspect_raw.py': "import argparse\nimport json\nfrom pathlib import Path\nfrom pyspark.sql import SparkSession\nfrom ingest_file import COLUMNS, digest\n\nap = argparse.ArgumentParser()\nap.add_argument('--manifest', required=True)\na = ap.parse_args()\nm = json.loads(Path(a.manifest).read_text(encoding='utf-8'))\np = Path(m['destination'])\nif (m['status'] != 'success' or not p.is_file() or p.stat().st_size != m['bytes']\n        or digest(p) != m['sha256']):\n    raise ValueError('raw/manifest validation failed')\nspark = SparkSession.builder.master('local[2]').appName('nhom3-raw-inspect').getOrCreate()\ntry:\n    df = spark.read.option('header', True).option('mode', 'FAILFAST').csv(str(p.resolve()))\n    if df.columns != COLUMNS:\n        raise ValueError('wrong columns')\n    rows = df.count()\n    if rows == 0:\n        raise ValueError('empty raw')\n    # count alone may prune CSV columns, so force reading every field as well.\n    def consume_partition(records):\n        n = 0\n        for row in records:\n            tuple(row)\n            n += 1\n        yield n\n    parsed = df.rdd.mapPartitions(consume_partition).sum()\n    if parsed != rows:\n        raise ValueError('parsed rows/count mismatch')\n    print('RAW PATH:', p)\n    print('ROWS:', rows)\n    df.printSchema()\n    df.show(5, truncate=False)\n    out = dict(run_id=m['run_id'], destination=str(p), sha256=m['sha256'],\n               rows=rows, parsed_rows=parsed, columns=df.columns,\n               spark_version=spark.version, status='PASS')\n    report = Path('metadata') / ('spark_read_' + m['run_id'] + '.json')\n    report.parent.mkdir(exist_ok=True)\n    report.write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding='utf-8')\n    print('SPARK RAW PASS', report)\nfinally:\n    spark.stop()\n", 'make_sample.py': "import argparse\nimport csv\nimport json\nfrom pathlib import Path\nfrom ingest_file import COLUMNS, digest\n\nap = argparse.ArgumentParser()\nap.add_argument('--source', required=True)\na = ap.parse_args()\nsource = Path(a.source).resolve()\ntarget = Path('data/sample/paysim_sample.csv')\ntarget.parent.mkdir(parents=True, exist_ok=True)\nwith source.open(encoding='utf-8-sig', newline='') as f:\n    reader = csv.reader(f)\n    header = next(reader)\n    if header != COLUMNS:\n        raise ValueError('header mismatch')\n    with target.open('x', encoding='utf-8', newline='') as out:\n        writer = csv.writer(out)\n        writer.writerow(header)\n        count = 0\n        for row in reader:\n            writer.writerow(row)\n            count += 1\n            if count == 1000:\n                break\nPath('metadata').mkdir(exist_ok=True)\nPath('metadata/sample_lineage.json').write_text(json.dumps(dict(\n    source=str(source), parent_sha256=digest(source), sample=str(target),\n    sample_sha256=digest(target), rows=count,\n    method='first 1000 data rows, CSV re-serialized; derived sample, not full raw'), indent=2), encoding='utf-8')\nprint('SAMPLE CREATED', count, target)\n", 'verify_manifest.py': "import argparse\nimport json\nfrom pathlib import Path\nfrom ingest_file import digest\n\nap = argparse.ArgumentParser()\nap.add_argument('--manifest', required=True)\na = ap.parse_args()\nm = json.loads(Path(a.manifest).read_text(encoding='utf-8'))\nif m['status'] != 'success':\n    raise ValueError('not a success manifest')\np = Path(m['destination'])\nif not p.is_file() or p.stat().st_size <= 0 or p.stat().st_size != m['bytes']:\n    raise ValueError('raw file/size mismatch')\nif digest(p) != m['sha256']:\n    raise ValueError('SHA-256 mismatch')\nprint('MANIFEST PASS', p, m['sha256'])\n"}
for name, content in SCRIPT_FILES.items():
    (PROJECT/'src'/name).write_text(content,encoding='utf-8')

def run_job(script, args=(), log_name=None, expected_code=0):
    command=[sys.executable,str(PROJECT/'src'/script),*map(str,args)]
    proc=subprocess.run(command,cwd=PROJECT,capture_output=True,text=True,encoding='utf-8')
    combined='$ '+' '.join(command)+'\n'+proc.stdout+'\n'+proc.stderr
    if log_name:
        (PROJECT/'logs'/log_name).write_text(combined,encoding='utf-8')
    print(combined[-16000:])  # full log is in logs, console preview is capped
    if proc.returncode != expected_code:
        raise RuntimeError(f'{script}: exit={proc.returncode}, expected={expected_code}. Mo log day du')
    return proc

def ingest(source, dataset, kind='full_snapshot'):
    proc=run_job('ingest_file.py',['--source',source,'--dataset',dataset,'--source-kind',kind])
    lines=[line for line in proc.stdout.splitlines() if line.startswith('SUCCESS manifest=')]
    if len(lines)!=1:
        raise RuntimeError('Chua nhan duoc manifest SUCCESS')
    return Path(lines[0].split('manifest=',1)[1])
print('SCRIPTS READY', sorted(SCRIPT_FILES))


## Ô 06 chạy smoke test Spark


In [ ]:
run_job('env_check.py', log_name='env_check.txt')
versions = dict(python=platform.python_version(), pyspark=version('pyspark'),
    java=java_info(selected)[1], mode='local[2] on Kaggle CPU',
    recorded_at_utc=datetime.now(timezone.utc).isoformat())
for tag, path in [('kaggle_git_commit','/etc/git_commit'),('kaggle_build_date','/etc/build_date')]:
    versions[tag]=Path(path).read_text().strip() if Path(path).exists() else None
(PROJECT/'metadata/environment.json').write_text(json.dumps(versions,indent=2),encoding='utf-8')
freeze=subprocess.run([sys.executable,'-m','pip','freeze'],capture_output=True,text=True,check=True)
(PROJECT/'metadata/kaggle_packages_freeze.txt').write_text(freeze.stdout,encoding='utf-8')
print('ENV evidence saved. pip freeze is an inventory, not an instruction to reinstall the entire Kaggle image.')


## Ô 07 tạo sample rồi chạy toàn chuỗi sample


In [ ]:
sample=PROJECT/'data/sample/paysim_sample.csv'
if not sample.exists():
    run_job('make_sample.py',['--source',SOURCE],log_name='make_sample.txt')
else:
    lineage=json.loads((PROJECT/'metadata/sample_lineage.json').read_text())
    if lineage['parent_sha256']!=SOURCE_HASH or sha256_file(sample)!=lineage['sample_sha256']:
        raise RuntimeError('Sample cu khac snapshot. Dung project moi, khong sua raw cu')
SAMPLE_MANIFEST=ingest(sample,'paysim_sample','derived_sample')
sample_id=json.loads(SAMPLE_MANIFEST.read_text())['run_id']
run_job('verify_manifest.py',['--manifest',SAMPLE_MANIFEST],f'verify_sample_{sample_id}.txt')
run_job('inspect_raw.py',['--manifest',SAMPLE_MANIFEST],f'spark_sample_{sample_id}.txt')
sample_report=json.loads((PROJECT/'metadata'/f'spark_read_{sample_id}.json').read_text())
lineage=json.loads((PROJECT/'metadata/sample_lineage.json').read_text())
assert sample_report['rows']==lineage['rows']
print('SAMPLE CHAIN PASS',sample_report['rows'])


## Ô 08 ingest full CSV và đọc bằng Spark

Phải có ENV và sample PASS trước. Không đọc toàn CSV bằng pandas. Không collect/toPandas toàn dữ liệu. Raw giữ nguyên byte, kể cả nhãn và số dư bằng 0.


In [ ]:
assert shutil.disk_usage(PROJECT).free > 3*1024**3, 'Can it nhat 3 GiB trong cho raw/temp'
FULL_MANIFEST=ingest(SOURCE,'paysim')
full_meta=json.loads(FULL_MANIFEST.read_text())
run_job('verify_manifest.py',['--manifest',FULL_MANIFEST],f'verify_full_{full_meta["run_id"]}.txt')
run_job('inspect_raw.py',['--manifest',FULL_MANIFEST],f'spark_full_{full_meta["run_id"]}.txt')
FULL_REPORT=json.loads((PROJECT/'metadata'/f'spark_read_{full_meta["run_id"]}.json').read_text())
assert full_meta['sha256']==SOURCE_HASH
print('FULL ROWS:',FULL_REPORT['rows'])
print('FULL BYTES:',full_meta['bytes'])
print('FULL MANIFEST:',FULL_MANIFEST)
# The local snapshot previously audited by the group has 6,362,620 rows.
# If Kaggle mounts a different hash/version, investigate; never edit raw to force a count.


## Ô 09 kiểm chạy cùng nguồn ba lần

Giữ run full ở ô 08, tạo thêm hai run. Ba run phải khác ID, cùng bytes/hash/count. Đây là version theo lần chạy, chưa dedup snapshot. Chạy lại ô 08/09 trong cùng session sẽ tạo thêm run theo đúng chính sách.


In [ ]:
FULL_RUNS=[FULL_MANIFEST]
for _ in range(2):
    FULL_RUNS.append(ingest(SOURCE,'paysim'))
rerun=[]
for index,manifest in enumerate(FULL_RUNS):
    m=json.loads(manifest.read_text())
    rid=m['run_id']
    run_job('verify_manifest.py',['--manifest',manifest],f'verify_rerun_{rid}.txt')
    if index>0:
        run_job('inspect_raw.py',['--manifest',manifest],f'spark_rerun_{rid}.txt')
    report=json.loads((PROJECT/'metadata'/f'spark_read_{rid}.json').read_text())
    rerun.append(dict(run_id=rid,bytes=m['bytes'],sha256=m['sha256'],
        destination=m['destination'],manifest=str(manifest),Spark_rows=report['rows']))
assert len({m['run_id'] for m in rerun})==3
assert len({(m['bytes'],m['sha256'],m['Spark_rows']) for m in rerun})==1
with (PROJECT/'docs/rerun_results.csv').open('w',newline='',encoding='utf-8') as f:
    writer=csv.DictWriter(f,fieldnames=list(rerun[0]));writer.writeheader();writer.writerows(rerun)
print('RERUN PASS: 3 separate runs, same snapshot')


## Ô 10 kiểm lỗi nguồn và phát hiện raw bị thay đổi bằng bản copy thử


In [ ]:
tests=[]
header=','.join(COLS)+'\n'
for name,text in [('empty.csv',''),('header_only.csv',header),('wrong.csv','a,b\n1,2\n'),('html.csv','<html>Login</html>')]:
    path=PROJECT/'fixtures'/name;path.write_text(text,encoding='utf-8')
    proc=run_job('ingest_file.py',['--source',path,'--dataset','paysim'],f'failure_{name}.txt',expected_code=1)
    tests.append(dict(test=name,result='PASS',exit_code=proc.returncode))
proc=run_job('ingest_file.py',['--source',PROJECT/'fixtures/missing.csv','--dataset','paysim'],
             'failure_missing.txt',expected_code=1)
tests.append(dict(test='missing.csv',result='PASS',exit_code=proc.returncode))
# Only alter a private copy of the sample, never accepted raw.
tampered=PROJECT/'fixtures/tampered.csv'
payload=bytearray(sample.read_bytes())
payload[0]=ord('X') if payload[0]!=ord('X') else ord('Y')
tampered.write_bytes(payload)  # same length, so verification must detect hash mismatch
m=json.loads(SAMPLE_MANIFEST.read_text());m['destination']=str(tampered)
bad_manifest=PROJECT/'fixtures/manifest_tampered.json';bad_manifest.write_text(json.dumps(m),encoding='utf-8')
proc=run_job('verify_manifest.py',['--manifest',bad_manifest],'failure_verify_tampered.txt',expected_code=1)
tests.append(dict(test='verify_tampered_copy',result='PASS',exit_code=proc.returncode))
(PROJECT/'docs/failure_tests.json').write_text(json.dumps(tests,indent=2),encoding='utf-8')
print('NEGATIVE TESTS PASS',len(tests))


## Ô 11 tạo README và report nháp theo form để nhóm hoàn thiện


In [ ]:
(PROJECT/'data/raw/.gitkeep').touch(exist_ok=True)
(PROJECT/'.gitignore').write_text('data/raw/*\n!data/raw/.gitkeep\nfixtures/\n__pycache__/\n.env\nkaggle.json\n',encoding='utf-8')
readme = """# Nhom 3 Checkpoint 2 tren Kaggle
Import Nhom3_Buoi6_Kaggle.ipynb, Add Input ealaxi/paysim1, CPU None, Internet On.
Chay o 01 den 12. Python cua image Kaggle, PySpark 4.1.1, JDK17/21.
src/env_check.py: smoke Spark, range 1000000 count, ENV PASS.
src/ingest_file.py --source <CSV> --dataset paysim: ingest nguyen byte.
src/verify_manifest.py --manifest <manifest>: kiem bytes/hash.
src/inspect_raw.py --manifest <manifest>: read/count/schema/show.
Trong Kaggle da setup, mot lenh ENV: python src/env_check.py.
Raw: data/raw/paysim/ingest_date=UTC-date/run_<id>/CSV + manifest.json.
Moi lan success tao run moi; cung snapshot co cung hash, khong overwrite, chua dedup.
Chi doc mot destination, khong wildcard nhieu run.
Sample 1000 hang dau chi de thu flow, co sample_lineage, khong thay full.
Save Version -> Save & Run All, cho thanh cong, mo ban da luu va kiem Outputs.
Download ho so nhe, du lieu raw de trong saved Notebook Output. Tai raw rieng neu can.
Kaggle Notebook version la bang chung chay, khong thay Git commit/tag theo rubric.
Tai notebook .ipynb da luu, dua code/docs/log/metadata/sample nho vao Git, tag checkpoint-2.
metadata/kaggle_packages_freeze.txt la inventory image, khong pip install toan bo file nay.
Source: https://www.kaggle.com/datasets/ealaxi/paysim1
Huong hoan thien docs/checkpoint_report.md voi Notebook version URL va Git commit that.
"""
(PROJECT/'README.md').write_text(readme,encoding='utf-8')
report=f"""# Checkpoint report Nhom 3
Ten nhom / thanh vien: Nhom 3. Hoang Thi Dieu Huong, Nguyen Thi Hang My, Nguyen Hoang Duong, Than Van Dung.
De tai: He thong phat hien gian lan tai chinh theo thoi gian thuc. Checkpoint nay la ENV, INGEST, RAW.
Git commit/tag: [Huong dien hash that sau khi dua code va evidence vao repo]
Kaggle Notebook version URL: [dien sau Save Version thanh cong]
Environment: Python {versions['python']}, Spark {versions['pyspark']}, Java xem metadata/environment.json. Mode local[2] tren Kaggle CPU.
Data source: ealaxi/paysim1, {SOURCE}, SHA256 {SOURCE_HASH}, version {KAGGLE_DATASET_VERSION}.
Ingestion command: python src/ingest_file.py --source {SOURCE} --dataset paysim
Raw location: {full_meta['destination']}
Manifest/log location: {FULL_MANIFEST}; logs/env_check.txt; logs/ingest_{full_meta['run_id']}.txt; logs/spark_full_{full_meta['run_id']}.txt
Rerun behavior: 3 run rieng, cung hash/bytes/count, khong overwrite, chua dedup snapshot.
Full data rows: {FULL_REPORT['rows']}; bytes: {full_meta['bytes']}.
Blocker con lai: [nhom dien thuc te, khong tu danh dau xong]
Owner + deadline: [dien neu con blocker]

## Exit ticket
1. Mot lenh ENV sau setup: python src/env_check.py
2. Version/snapshot: source URL, Input version neu biet, bytes va SHA256.
3. Chay 3 lan: 3 run moi, cung hash neu nguon khong doi. Spark doc 1 run.
4. Rui ro truoc Buoi 7: [nhom chot rui ro thuc te, owner, deadline].

Thuc hien Save Version va kiem Outputs truoc khi nop. Chuyen noi dung da xac nhan sang muc 4 Brief goc.
"""
(PROJECT/'docs/checkpoint_report.md').write_text(report,encoding='utf-8')
(PROJECT/'metadata/demo_run.json').write_text(json.dumps(dict(manifest=str(FULL_MANIFEST),
    destination=full_meta['destination'],sha256=SOURCE_HASH),indent=2),encoding='utf-8')
print(report)


## Ô 12 đóng hồ sơ nhẹ và kiểm Outputs

ZIP chỉ gồm code, tài liệu, log, metadata, sample nhỏ. Ba CSV full vẫn nằm riêng dưới `nhom3_buoi6/data/raw` để lưu bằng Notebook version. Không tải 1,5 GB raw về laptop nếu chưa cần.


In [ ]:
# Do not publish completion unless the evidence generated in this session agrees.
assert 'ENV PASS' in (PROJECT/'logs/env_check.txt').read_text()
assert sample_report['status']=='PASS' and FULL_REPORT['status']=='PASS'
assert sample_report['rows']==1000
assert len(rerun)==3 and len({r['run_id'] for r in rerun})==3
assert len({(r['bytes'],r['sha256'],r['Spark_rows']) for r in rerun})==1
assert all(t['result']=='PASS' for t in tests) and len(tests)==6
man_dir=PROJECT/'metadata/raw_manifests';man_dir.mkdir(exist_ok=True)
for manifest_path in [SAMPLE_MANIFEST]+FULL_RUNS:
    record=json.loads(manifest_path.read_text())
    (man_dir/(record['run_id']+'.json')).write_text(json.dumps(record,indent=2),encoding='utf-8')
contract=dict(schema_version=2,dataset='paysim',source_sha256=SOURCE_HASH,
    bytes=full_meta['bytes'],full_rows=FULL_REPORT['rows'],raw_destination=full_meta['destination'],
    manifest=str(FULL_MANIFEST),checkpoint_session=RUN_ID,
    source_url=source_meta['source_url'],spark_read_status='PASS')
(PROJECT/'metadata/transfer_contract.json').write_text(json.dumps(contract,indent=2),encoding='utf-8')
Path('/kaggle/working/Nhom3_Checkpoint_to_Train.json').write_text(json.dumps(contract,indent=2),encoding='utf-8')
status=dict(status='PASS',session=RUN_ID,environment='PASS',sample_read='PASS',
    full_read='PASS',rerun='PASS',negative_tests='PASS',source_sha256=SOURCE_HASH,
    full_rows=FULL_REPORT['rows'],git_commit='Group must fill actual repository commit after upload')
(PROJECT/'metadata/pipeline_status.json').write_text(json.dumps(status,indent=2),encoding='utf-8')
from zipfile import ZipFile, ZIP_DEFLATED
archive=Path('/kaggle/working/Nhom3_Buoi6_ho_so_nhe.zip')
with ZipFile(archive,'w',ZIP_DEFLATED) as z:
    for file in PROJECT.rglob('*'):
        if not file.is_file():
            continue
        rel=file.relative_to(PROJECT)
        if rel.parts[:2]==('data','raw') or rel.parts[0] in {'fixtures','.git'} or '__pycache__' in rel.parts:
            continue
        z.write(file,rel.as_posix())
with ZipFile(archive) as z:
    assert z.testzip() is None
print('LIGHT ARCHIVE:',archive,archive.stat().st_size,'bytes')
print('Raw saved separately at:',PROJECT/'data/raw')
print('NEXT: Save Version -> Save & Run All -> wait for success -> open saved version Outputs.')

print('CHECKPOINT PIPELINE PASS:', json.dumps(status))


## Sau khi chạy
Save Version → Save & Run All, chờ thành công và kiểm chính saved version.
Tải Nhom3_Buoi6_ho_so_nhe.zip và Nhom3_Checkpoint_to_Train.json. Raw full ở saved Outputs.
ZIP nhẹ có bản sao manifest ở metadata/raw_manifests, không có CSV full.
Chép code/log/metadata/notebook về repo, điền commit nhóm thật và URL notebook version vào report.
Mỗi session mới có run ID mới. Đừng trộn đường dẫn giữa phiên tương tác và phiên Save & Run All.
Notebook train v2 có thể đọc PaySim trực tiếp và kiểm đúng hash/bytes; không phải upload lại CSV.
